In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERREVIEWS_ENRICHED'

In [0]:
orderreviews_df = spark.table('SALES.BRONZE.ORDERREVIEWS')
orderreviews_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

orderreviews_enriched_df = orderreviews_df.select(
trim(col("review_id")).alias("review_id"),

trim(col("order_id")).alias("order_id"),

col("review_score").cast("int").alias("review_score"),

trim(col("review_comment_title")).alias("review_comment_title"),

trim(col("review_comment_message")).alias("review_comment_message"),

to_timestamp(col("review_creation_date")).alias("review_creation_date"),

to_timestamp(col("review_answer_timestamp")).alias("review_answer_timestamp"),

col("file_name").alias("file_name"),

col("file_path").alias("file_path"),

current_timestamp().alias('load_timestamp')

).filter('review_id IS NOT NULL')

display(orderreviews_enriched_df.limit(1))

In [0]:
flagged_df = (orderreviews_enriched_df.select("review_id","order_id","review_score","review_comment_title","review_comment_message","review_creation_date","review_answer_timestamp","file_name","file_path","load_timestamp")
.withColumn("is_review_score_positive", col("review_score") > 0))

flagged_df = flagged_df.where("review_comment_title IS NOT NULL OR review_comment_message IS NOT NULL")

display(flagged_df)

In [0]:
flagged_df.createOrReplaceTempView('enriched_orderreviews_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY review_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_orderreviews_temp_vw
  )
  WHERE rn = 1
) src
ON (src.review_id  = tgt.review_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.review_id = src.review_id,
    tgt.order_id = src.order_id,
    tgt.review_score = src.review_score,
    tgt.review_comment_title = src.review_comment_title,
    tgt.review_comment_message = src.review_comment_message,
    tgt.review_creation_date = src.review_creation_date,
    tgt.review_answer_timestamp = src.review_answer_timestamp,
    tgt.is_review_score_positive = src.is_review_score_positive
    
WHEN NOT MATCHED THEN
  INSERT (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp,
    is_review_score_positive,
    file_name,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.review_id,
    src.order_id,
    src.review_score,
    src.review_comment_title,
    src.review_comment_message,
    src.review_creation_date,
    src.review_answer_timestamp,
    src.is_review_score_positive,
    src.file_name,
    src.file_path,
    current_timestamp()
  )
""")
